# H04 site inference

Load the saved pooled H04 model and generate a score and a binary prediction per RNA site.

1. Select the repository's `.venv` Python kernel.
2. Change `INPUT_PATH` in **Settings** to your new dataset.
3. Choose **Restart Kernel and Run All**.

The default input is `data/raw/dataset2.json.gz`. The label file is not used.
For AWS Ubuntu setup with VS Code Remote SSH, follow [README.md](../README.md).
Inference does not train or tune the model. Each run saves two CSVs and one metadata JSON.


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import csv
import json
import sys

import pandas as pd
from IPython.display import display

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src/inference.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the project root or notebooks/ folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.inference import load_model_bundle, predict_to_csv

print("Project:", ROOT)


Project: /Users/jiacheng/Documents/GitHub/DSA4262_Genomies


## Settings

- `MODEL_PATH`: the preserved H04 model; no training run directory is required.
- `INPUT_PATH`: your input file, relative to `ROOT` or an absolute path.
- `INPUT_KIND`: `"reads"` for project JSON/JSONL (optionally gzip) or per-read
  Parquet; `"features"` for prepared site-feature CSV/Parquet.
- `OUTPUT_DIR`: destination for scores and metadata.

Each run creates three files in `OUTPUT_DIR`:

- `Genomies_<dataset name>.csv`: `transcript_id`, `transcript_position`, `score`.
- `Genomies_<dataset name>_pred.csv`: the same three columns plus `prediction`.
- `Genomies_<dataset name>.metadata.json`: provenance and both CSV layouts.

For example, `dataset2.json.gz` produces `Genomies_dataset2.csv` and
`Genomies_dataset2_pred.csv`. Existing outputs are preserved; move them or choose
another `OUTPUT_DIR` before rerunning the same dataset.

For raw input, each site needs a seven-base A/C/G/T sequence and all its reads,
each containing the same nine signal measurements used in training. Labels and
gene IDs are unnecessary. See `README.md` for the exact format.


In [ ]:
# ----- EDIT THESE SETTINGS -----
MODEL_PATH = ROOT / "models/h04.joblib"
INPUT_PATH = ROOT / "data/raw/dataset2.json.gz"  # Change to your input file
INPUT_KIND = "reads"  # "reads" or "features"

OUTPUT_DIR = ROOT / "outputs"
BATCH_SIZE = 200_000
MODEL_THREADS = 4
# ----- END SETTINGS -----

print("Model:", MODEL_PATH)
print("Input:", INPUT_PATH)
print("Input kind:", INPUT_KIND)


Model: /Users/jiacheng/Documents/GitHub/DSA4262_Genomies/models/h04.joblib
Input: /Users/jiacheng/Documents/GitHub/DSA4262_Genomies/data/finalLeaderboard/dataset2.json.gz
Input kind: reads


## Load H04 and generate scores

The model should report **H04: hgb (engineered, none)**, from pooled run
`20261006T085253_d868f3c6`. Its validation threshold is approximately **0.208393**;
the `_pred.csv` contains both continuous scores and labels obtained using this threshold.

Raw reads are processed in batches with each site's reads kept together. Feature
calculations reuse the training code, and input columns are matched to the model.


In [3]:
bundle = load_model_bundle(MODEL_PATH)
print("Model:", bundle.get("model_name", type(bundle["model"]).__name__))
print("Training run:", bundle.get("run_id"))
print("Validation threshold:", bundle.get("threshold"))

input_name = Path(INPUT_PATH)
if input_name.suffix.lower() == ".gz":
    input_name = input_name.with_suffix("")
dataset_name = input_name.stem
OUTPUT_PATH = Path(OUTPUT_DIR).expanduser().resolve() / f"Genomies_{dataset_name}.csv"
PREDICTION_PATH = OUTPUT_PATH.with_name(f"{OUTPUT_PATH.stem}_pred.csv")
METADATA_PATH = OUTPUT_PATH.with_suffix(".metadata.json")
SCORE_COLUMNS = ["transcript_id", "transcript_position", "score"]
PREDICTION_COLUMNS = [*SCORE_COLUMNS, "prediction"]

for path in (OUTPUT_PATH, PREDICTION_PATH, METADATA_PATH):
    if path.exists():
        raise FileExistsError(f"Move the existing output or change OUTPUT_DIR: {path}")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Score once, then export the exact column layouts without rounding the scores.
with TemporaryDirectory(prefix=".genomies-", dir=OUTPUT_PATH.parent) as temporary_dir:
    temporary_dir = Path(temporary_dir)
    scored_path = temporary_dir / "scored.csv"
    run_metadata = predict_to_csv(
        MODEL_PATH, INPUT_PATH, scored_path,
        input_kind=INPUT_KIND, batch_size=BATCH_SIZE, threads=MODEL_THREADS,
    )
    score_export = temporary_dir / OUTPUT_PATH.name
    prediction_export = temporary_dir / PREDICTION_PATH.name
    with (
        scored_path.open(encoding="utf-8", newline="") as source,
        score_export.open("x", encoding="utf-8", newline="") as score_file,
        prediction_export.open("x", encoding="utf-8", newline="") as prediction_file,
    ):
        score_writer = csv.DictWriter(score_file, fieldnames=SCORE_COLUMNS, extrasaction="ignore")
        prediction_writer = csv.DictWriter(prediction_file, fieldnames=PREDICTION_COLUMNS, extrasaction="ignore")
        score_writer.writeheader()
        prediction_writer.writeheader()
        for row in csv.DictReader(source):
            score_writer.writerow(row)
            prediction_writer.writerow(row)

    run_metadata.update(
        output_path=str(OUTPUT_PATH),
        output_columns=SCORE_COLUMNS,
        identifier_columns=SCORE_COLUMNS[:2],
        prediction_output_path=str(PREDICTION_PATH),
        prediction_output_columns=PREDICTION_COLUMNS,
    )
    metadata_export = temporary_dir / METADATA_PATH.name
    metadata_export.write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
    score_export.rename(OUTPUT_PATH)
    prediction_export.rename(PREDICTION_PATH)
    metadata_export.rename(METADATA_PATH)

print(f"Scores CSV: {OUTPUT_PATH}")
print(f"Predictions CSV: {PREDICTION_PATH}")
print(f"Metadata: {METADATA_PATH}")


Model: H04: hgb (engineered, none)
Training run: 20261006T085253_d868f3c6
Validation threshold: 0.20839324867970055
Model: H04: hgb (engineered, none)
Input: /Users/jiacheng/Documents/GitHub/DSA4262_Genomies/data/finalLeaderboard/dataset2.json.gz (reads)
Scored 172 sites
Scored 474 sites
Scored 744 sites
Scored 953 sites
Scored 1,176 sites
Scored 1,323 sites
Saved 1,323 site scores and predictions to /Users/jiacheng/Documents/GitHub/DSA4262_Genomies/outputs/H04_dataset2_json_20261026T184511_d9ec5e6d.csv
